# Análise de Cancelamento de Clientes

**Python Insights** · Jornada Python (Hashtag Treinamentos) · Alessandro José dos Santos

### O case
Uma empresa com mais de 800 mil clientes percebeu que a maior parte da sua base já cancelou o serviço. O objetivo é entender **os principais motivos dos cancelamentos** e propor **as ações mais eficientes** para reduzir esse número.

A análise usa uma amostra de 50 mil clientes (`cancelamentos.csv`).

## Passo 1 — Importar a base de dados

In [ ]:
import pandas as pd

tabela = pd.read_csv("cancelamentos.csv")
display(tabela)

## Passo 2 — Visualizar e entender a base
A coluna `CustomerID` só identifica o cliente e não ajuda a explicar o cancelamento, então ela é removida.

In [ ]:
tabela = tabela.drop(columns="CustomerID")
display(tabela)

## Passo 3 — Corrigir os problemas da base
Algumas linhas têm valores vazios. Como são pouquíssimas (4 de 50.000), elas são excluídas.

In [ ]:
display(tabela.info())

tabela = tabela.dropna()

display(tabela.info())

## Passo 4 — Análise inicial: quantos clientes cancelaram?

In [ ]:
# 0 = não cancelou | 1 = cancelou
display(tabela["cancelou"].value_counts())
display(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

**Resultado:** 56,8% dos clientes cancelaram.

## Passo 5 — Análise detalhada: como cada característica influencia o cancelamento
Um histograma para cada coluna, separando quem cancelou de quem continuou.

In [ ]:
import plotly.express as px

for coluna in tabela.columns:
    grafico = px.histogram(tabela, x=coluna, color="cancelou", text_auto=True)
    grafico.show()

### Principais descobertas

| Causa | O que os dados mostram | Ação proposta |
|---|---|---|
| **Contrato mensal** | 100% dos clientes com contrato mensal cancelaram | Oferecer desconto para migrar para os planos trimestral e anual |
| **Ligações ao call center** | A partir de 5 ligações, praticamente todos cancelaram | Alerta quando o cliente ligar 3 vezes, para resolver o problema antes |
| **Atraso no pagamento** | Acima de 20 dias de atraso, 100% cancelaram | Alerta e negociação quando o atraso chegar a 15 dias |

Outro ponto de atenção: **todos os clientes acima de 50 anos cancelaram**, o que vale uma investigação específica sobre a experiência desse público.

## Passo 6 — Simulação: quanto o cancelamento cai se as causas forem resolvidas
Filtramos a base mantendo só os clientes que **não** se encaixam em cada causa, de forma acumulada.

In [ ]:
# 1. Sem contrato mensal
tabela = tabela[tabela["duracao_contrato"] != "Monthly"]

# 2. Até 4 ligações ao call center
tabela = tabela[tabela["ligacoes_callcenter"] <= 4]

# 3. Até 20 dias de atraso no pagamento
tabela = tabela[tabela["dias_atraso"] <= 20]

display(tabela["cancelou"].value_counts(normalize=True).map("{:.1%}".format))

## Conclusão

Atacando as três causas principais, a taxa de cancelamento **cai de 56,8% para 18,4%**.

| Cenário | Taxa de cancelamento |
|---|---|
| Situação atual | 56,8% |
| Sem contrato mensal | 46,1% |
| + até 4 ligações ao call center | 26,4% |
| + até 20 dias de atraso | **18,4%** |